In [ ]:
import json
logs_online_shared_view_first = "logs_online_shared_view_first/analysis/"
logs_online_non_shared_view_first = "logs_online_non_shared_view_first/analysis/"

with open(f"{logs_online_shared_view_first}conversations_20251212_161733.json", encoding="utf-8") as f:
    data1 = json.load(f)

sess_shared_view = set()
sess_non_shared_view = set()
for k, ex in data1.items():
    if len(ex["all_messages"]):
        if "2025-12-08" in ex["all_messages"][0]["timestamp"] and len(ex["puzzles"]) > 8:
            sess_shared_view.update([k])

print(list(sess_shared_view), len(sess_shared_view))

import json
with open(f"{logs_online_non_shared_view_first}conversations_20251210_164627.json", encoding="utf-8") as f:
    data2 = json.load(f)

for k, ex in data2.items():
    if len(ex["all_messages"]):
        if "2025-12-10" in ex["all_messages"][0]["timestamp"] and len(ex["puzzles"]) > 8:
            sess_non_shared_view.update([k])

data = data1 | data2
print(list(sess_non_shared_view), len(sess_non_shared_view), len(data))

In [ ]:
import csv
human_helper_shared = []
human_worker_shared = []
human_helper_non_shared = []
human_worker_non_shared = []
with open(f"{logs_online_shared_view_first}collaboration_metrics_20251212_161733.csv", encoding="utf-8") as f:
    metrics = csv.DictReader(f)
    for m in metrics:
        if m["session_id"] not in sess_shared_view:
            continue
        if m["helper_mode"] == "human" and int(m["total_messages"]) > 100:
            human_helper_shared.append(m["session_id"])
        if m["helper_mode"] == "llm" and int(m["total_messages"]) > 100:
            human_worker_shared.append(m["session_id"])

with open(f"{logs_online_non_shared_view_first}collaboration_metrics_20251210_164627.csv", encoding="utf-8") as f:
    metrics = csv.DictReader(f)
    for m in metrics:
        if m["session_id"] not in sess_non_shared_view:
            continue
        if m["helper_mode"] == "human" and int(m["total_messages"]) > 100:
            human_helper_non_shared.append(m["session_id"])
        if m["helper_mode"] == "llm" and int(m["total_messages"]) > 100:
            human_worker_non_shared.append(m["session_id"])

In [ ]:
print(len(human_helper_shared), len(human_worker_shared), len(human_helper_non_shared), len(human_worker_non_shared))

In [ ]:
# # human_helper = ["a20f4c15-e3dc-4711-afd6-fa57321290ba"]
# ex = data["a20f4c15-e3dc-4711-afd6-fa57321290ba"]
# data[human_helper[2]]["puzzles"]["0"][0]
# m["all_messages"][0]
# human_helper
with open("sessions_12_08_and_10_2025.json", "w", encoding="utf-8") as f:
    json.dump({session_id: data[session_id] for session_id in human_helper_shared + human_worker_shared + human_helper_non_shared + human_worker_non_shared}, f)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def print_session(human_sessions, aggr="sum", role="human_helper", scope="Human", flag=False):
    l1 = []
    l2 = []
    p_idx = []
    sessions = []
    rows = []
    if role == "human_helper":
        roles = {"helper": "Human", "worker": "AI"}
    else:
        roles = {"helper": "AI", "worker": "Human"}
        
    worker = []
    for session_id in human_sessions:
        d = data[session_id]["puzzles"]
        i = 0
        for p, ex in d.items():

            if not flag:
                if p == "null" or p in ["6", "7", "8", "9"]:
                    # print(p, len(ex))
                    continue
            else:
                if p == "null" or p in ["1", "2", "3", "4", "5"]:
                    # print(p, len(ex))
                    continue
            last = "none"
            worker = []
            helper = []
            ms = []
            for i, e in enumerate([e_tmp for e_tmp in ex["messages"] if e_tmp["message"]]):
                if i > 0 and "The board is empty" in e["message"]:
                        continue
                if last != e["sender"]:
                    last = e["sender"]
                    message = e["message"]
                else:
                    message = message + "\n" + e["message"]

                if roles[e["sender"]] == scope:
                    ms.append(len(message))
                    
            if aggr == "sum":
                if ex["puzzle_details"]["config"]["puzzle_group"] == "Shared view":
                    l1.append(np.sum(ms))
                else:
                    l2.append(np.sum(ms))
            elif aggr == "mean":
                if ex["puzzle_details"]["config"]["puzzle_group"] == "Shared view":
                    l1.append(np.mean(ms))
                else:
                    l2.append(np.mean(ms))
            else:
                if ex["puzzle_details"]["config"]["puzzle_group"] == "Shared view":
                    l1.append(len(ms))
                else:
                    l2.append(len(ms))
            
            p_idx.append(p)
            sessions.append(session_id)
        # print(len(p_idx), len(l1), len(l2), len(sessions), session_id)
        
    end = min(len(l1), len(l2))

    df = pd.DataFrame.from_dict({f"shared view": l1[:end], f"non-shared view": l2[:end], "puzzle": p_idx[:end], "session": sessions[:end]})
    g = df.groupby('puzzle')  
      
    mean = g[[f"shared view",f"non-shared view"]].mean()  
    std  = g[[f"shared view",f"non-shared view"]].std()

    role_str = role.replace("_", " ")
    ax = mean.plot()#yerr=std)  
    x = ["practice", "1", "2", "3", "4"]
    ax.set_xticks([0,1,2,3,4]) 
    ax.set_xticklabels(x) 
    ax.set_xlabel(f'Puzzle with {role_str}') 
    if roles["helper"] == scope:
        scope_str = f"{scope} helper"
    else:
        scope_str = f"{scope} worker"
    if aggr == "len":
        ax.set_ylabel(f'{scope_str} communication (num of turns)') 
    else:
        ax.set_ylabel(f'{scope_str} communication ({aggr} num character)') 

    if not flag:
        ax.axvspan(0, 1, color='yellow', alpha=0.15, label='practice') 

    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    
    plt.savefig(f"{role}_{scope}_messages_{aggr}.png", dpi=300, bbox_inches="tight")  

print_session(human_helper_shared+human_helper_non_shared, aggr="sum", role="human_helper")
# print_session(human_helper_shared+human_helper_non_shared, aggr="sum", role="human_helper", flag=True)
print_session(human_worker_shared+human_worker_non_shared, aggr="sum", role="human_worker")

# print_session(human_helper_shared+human_helper_non_shared, aggr="mean", role="AI_helper")
# print_session(human_worker_shared+human_worker_non_shared, aggr="mean", role="human_helper", scope="AI")

print_session(human_helper_shared+human_helper_non_shared, aggr="sum", role="human_helper", scope="AI")
print_session(human_worker_shared+human_worker_non_shared, aggr="sum", role="human_worker", scope="AI")

In [ ]:
# =========================================  
# COMPLETE END-TO-END ANALYSIS CELL (sum(ms))  
# Only requirement: `data` dictionary exists  
# =========================================  
  
import pandas as pd  
import numpy as np  
import matplotlib.pyplot as plt  
from scipy.stats import ttest_rel  
import statsmodels.formula.api as smf  
  
# ------------------------------------  
# 1. EXTRACT sum(ms) FROM RAW DATA  
# ------------------------------------  
rows = []  
  
roles = {"helper": "Human", "worker": "AI"}  # adjust if roles are reversed  
scope = "Human"                              # analyze Human messages only  

data_new = {session_id: data[session_id] for session_id in human_helper_shared + human_worker_shared + human_helper_non_shared + human_worker_non_shared}
# data_worker = {session_id: data[session_id] for session_id in human_helper_shared + human_worker_shared + human_helper_non_shared + human_worker_non_shared}
for session_id, session_data in data_new.items():  
    puzzles = session_data.get("puzzles", {})  
  
    for p, ex in puzzles.items():  
  
        # Skip invalid puzzles (match your original code)  
        if p == "null" or p in ["6", "7", "8", "9"]:  
            continue  
  
        last_sender = None  
        message = ""  
        ms = []  
  
        for e in [m for m in ex.get("messages", []) if m.get("message")]:  
            if "The board is empty" in e["message"]:  
                continue  
  
            if e["sender"] != last_sender:  
                message = e["message"]  
                last_sender = e["sender"]  
            else:  
                message += "\n" + e["message"]  
  
            # if roles.get(e["sender"]) == scope:  
            ms.append(len(message))  
  
        rows.append({  
            "session": session_id,  
            "puzzle": str(p),  
            "view": (  
                "shared"  
                if ex["puzzle_details"]["config"]["puzzle_group"] == "Shared view"  
                else "non-shared"  
            ),  
            "sum_ms": 1 if np.sum(ms) == 0 else np.sum(ms)
        })  
  
# ------------------------------------  
# 2. BUILD DATAFRAME  
# ------------------------------------  
df_long = pd.DataFrame(rows)  
  
from scipy.stats import mannwhitneyu  
  
df_session = (  
    df_long  
    .groupby(["session", "view"])["sum_ms"]  
    .mean()  
    .reset_index()  
)  
  
shared = df_session[df_session["view"] == "shared"]["sum_ms"]  
non_shared = df_session[df_session["view"] == "non-shared"]["sum_ms"]  
  
u, p = mannwhitneyu(non_shared, shared, alternative="less")  
  
print(f"Mann–Whitney U: p = {p:.5f}, U = {u}")  
print("Median shared:", shared.median())  
print("Median non-shared:", non_shared.median())   

import seaborn as sns  
import matplotlib.pyplot as plt  
  
view_order = ["shared", "non-shared"]  
  
sns.boxplot(  
    data=df_session,  
    x="view",  
    y="sum_ms",  
    order=view_order,  
    showfliers=False  
)  
  
sns.stripplot(  
    data=df_session,  
    x="view",  
    y="sum_ms",  
    order=view_order,  
    color="black",  
    alpha=0.5  
)  
  
plt.xlabel("")  
plt.ylabel("Total communication (sum num character)")  
plt.tight_layout()  
# plt.show()  
plt.savefig(f"total_communication.png", dpi=300, bbox_inches="tight")

In [ ]:
from azure.identity import (
    AzureCliCredential,
    ChainedTokenCredential,
    DefaultAzureCredential,
    get_bearer_token_provider,
)
import requests
import base64 
def gpt_request_chat(messages: list, 
                model: str = "gpt-4o_2024-08-06", 
                max_tokens: int = 1500,
                api_version: str = "2025-01-01-preview") -> str:
  """Make a request to Azure OpenAI's GPT model."""

  try:
    base_url = "https://trapi.research.microsoft.com/"
    trapi_endpoint = "msrc/shared/"
    url = f'{base_url}{trapi_endpoint}openai/deployments/{model}/chat/completions?api-version={api_version}'
    azure_credential = ChainedTokenCredential(
        AzureCliCredential(),
        DefaultAzureCredential(),
    )
    token_provider = get_bearer_token_provider(azure_credential, "api://trapi/.default")
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token_provider()}"
    }
    data = {  
        "messages": messages
    }
    nextResponse = requests.post(url, headers=headers, json=data)
    return nextResponse.json()["choices"][-1]["message"]["content"]
  except Exception as e:
    print(e)
    # print(nextResponse)
    return f"GENERATION ERROR: {str(e)}"

def gpt_request(prompt: str, 
                model: str = "gpt-4o_2024-08-06", 
                max_tokens: int = 1500,
                api_version: str = "2025-01-01-preview") -> str:
  """Make a request to Azure OpenAI's GPT model."""

  try:
    base_url = "https://trapi.research.microsoft.com/"
    trapi_endpoint = "msrc/shared/"
    url = f'{base_url}{trapi_endpoint}openai/deployments/{model}/chat/completions?api-version={api_version}'
    azure_credential = ChainedTokenCredential(
        AzureCliCredential(),
        DefaultAzureCredential(),
    )
    token_provider = get_bearer_token_provider(azure_credential, "api://trapi/.default")
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token_provider()}"
    }
    data = {  
        "messages": [{"role": "user", "content": prompt}]
    }
    nextResponse = requests.post(url, headers=headers, json=data)
    return nextResponse.json()["choices"][-1]["message"]["content"]
  except Exception as e:
    print(e)
    # print(nextResponse)
    return f"GENERATION ERROR: {str(e)}"

import ast
def get_json(response, default={}):  
    """  
    Extract JSON content from a response string.  
    """  
    response = response.replace("\n", " ").replace("\t", " ").replace("\'", "'")
    b_start = response.find("{")
    start = response.find("```json")  
    if start == -1:  
        start = response.find("```")
        if start > b_start:
            end = response.rfind("}")
            # print(response[b_start:end+1], start, end)
            try:
                return ast.literal_eval(response[b_start:end+1])
            except Exception as e:
                return default
        
        if start == -1:  
            start = 0  
        else:  
            start += len("```")
    else:  
        start += len("```json")  
  
    end = response.rfind("```", start)  
    if end == -1:  
        end = len(response)  

    if not response[start:end].strip().endswith("}"):
        # print(response[start:end], start, end)
        try: 
            # print(response[start:end], start, end)
            return ast.literal_eval(response[start:end] + "\n}")
        except Exception as e:
            return default

    try:  
    # print(response[start:end], start, end)
        return ast.literal_eval(response[start:end])  
    except Exception as e:
        if len(default) > 0:
            key = list(default.keys())[0]
            start = response.find(f"\"{key}\": ") + len(f"\"{key}\": ") + 1
            if start == -1:
                return {key: response}
                
            end = start + response[start:].find("\"")
            return {key: response[start:end]}
            
        print(e)
        return default  

def gpt_request_text_and_image_upload(prompt: str,  
                                      image_path: str,  
                                      model: str = "gpt-4o_2024-08-06",  
                                      max_tokens: int = 1500,  
                                      api_version: str = "2025-01-01-preview") -> str:  
    """  
    Make a request to Azure OpenAI's GPT model with both text and a locally uploaded image.  
    The image will be read from disk and sent as Base64 so the API can process it directly.  
    """  
    try:  
        # Read and Base64 encode the image  
        with open(image_path, "rb") as img_file:  
            image_base64 = base64.b64encode(img_file.read()).decode("utf-8")  
  
        base_url = "https://trapi.research.microsoft.com/"  
        trapi_endpoint = "msrc/shared/"  
        url = f'{base_url}{trapi_endpoint}openai/deployments/{model}/chat/completions?api-version={api_version}'  
        azure_credential = ChainedTokenCredential(  
            AzureCliCredential(),  
            DefaultAzureCredential(),  
        )  
        token_provider = get_bearer_token_provider(azure_credential, "api://trapi/.default")  
        headers = {  
            "Content-Type": "application/json",  
            "Authorization": f"Bearer {token_provider()}"  
        }  
  
        # Multimodal message with text + Base64 image  
        data = {  
            "messages": [  
                {  
                    "role": "user",  
                    "content": [  
                        {"type": "text", "text": prompt},
                        {"type": "image_url",
                        "image_url": {
                           "url": f"data:image/jpeg;base64,{image_base64}"
                        }
                        }
                    ]  
                }  
            ],  
            "max_tokens": max_tokens  
        }  
        response = requests.post(url, headers=headers, json=data)  
        response.raise_for_status()  
  
        return response.json()["choices"][-1]["message"]["content"]  
    except Exception as e:  
        print("Error:", e)  
        return f"GENERATION ERROR: {str(e)}"

In [ ]:
with open(f"{logs_online_shared_view_first}puzzle_task_performance.json", encoding="utf-8") as f:
    task_completions_shared = json.load(f)

with open(f"{logs_online_non_shared_view_first}puzzle_task_performance.json", encoding="utf-8") as f:
    task_completions_non_shared = json.load(f)

In [ ]:
def plot_performance(completions, view="shared view"):
    from collections import defaultdict
    session_to_results = defaultdict(list)
    for task in completions:
        for k, v in task.items():
            if k[-5] in ["6", "7", "8", "9"]:
                continue
            session_to_results[k[9:16]].append(v)

    from collections import Counter
    puzzle_to_summary = defaultdict(Counter)
    for k, v in session_to_results.items():
        for i, r in zip(["practice", "1", "2", "3", "4"], v):#enumerate(v):
            if r.endswith("1") or r.endswith("2"):
                r = r[:-2]
            puzzle_to_summary[i].update([r])

    import pandas as pd  
    import matplotlib.pyplot as plt   
      
    df = pd.DataFrame(puzzle_to_summary).T  
      
    custom_colors = ['orange', 'green', 'red']
    df[["no match", "partial match", "exact match"]].plot(  
        kind="bar",  
        stacked=True,  
        color=["red", "orange", "green"]  
    ) 

    
    plt.xticks(x, rotation=0)  
    plt.xlabel("Puzzle")  
    plt.ylabel("Count")  
    plt.title(f"Puzzle match results {view.replace('_', ' ')}")  
    plt.grid(axis="y", alpha=0.3) 
      
    plt.legend( bbox_to_anchor=(1.05, 1), loc="upper left")  
      
    plt.tight_layout()  
    plt.savefig(f"task_performances_{view}.png", dpi=300, bbox_inches="tight")  
    plt.show()   

# plot_performance(task_completions_shared, view="shared_view")
# plot_performance(task_completions_non_shared, view="non_shared_view")

In [ ]:
from collections import defaultdict
session_to_results = defaultdict(list)
rows = []
for task in task_completions_shared:
    for k, v in task.items():
        session_to_results[k[9:16]].append(v)

for session, performances in session_to_results.items():    
    for i in [1,2,3,4]:
        v = 0
        if performances[i] == "no match":
            v = 1
        rows.append({"view": "shared", "session": session, "puzzle": i, "success": v})

session_to_results = defaultdict(list)
for task in task_completions_non_shared:
    for k, v in task.items():
        session_to_results[k[9:16]].append(v)

for session, performances in session_to_results.items():    
    for i in [1,2,3,4]:
        v = 0
        if performances[i] == "no match":
            v = 1
        rows.append({"view": "non-shared", "session": session, "puzzle": i, "success": v})

# !pip install statsmodels  
import pandas as pd
import statsmodels.api as sm  
import statsmodels.formula.api as smf  
df_success = pd.DataFrame.from_dict(rows) 
  
# Keep only puzzles 2–5  

df_succ = df_success[df_success["puzzle"].isin([1, 2, 3, 4])].copy()  
df_succ["puzzle"] = df_succ["puzzle"].astype("category")  
# print(df_succ)
  
gee = smf.gee(  
    "success ~ C(view) * C(puzzle)",  
    groups="session",  
    data=df_succ,  
    family=sm.families.Binomial()  
)  
  
result = gee.fit()  
print(result.summary()) 
print(result.wald_test("C(view)[T.shared]:C(puzzle)[T.2] = 0, C(view)[T.shared]:C(puzzle)[T.3] = 0, C(view)[T.shared]:C(puzzle)[T.4] = 0"))
print(result.wald_test("C(view)[T.shared] = 0"))

In [ ]:
with open(f"{logs_online_shared_view_first}sessions_12_08_2025_annotations_new_refs_only_4.json", encoding="utf-8") as f:
    tmp = json.load(f)
    sessions_annotated_shared = {}
    for t in tmp:
        for k, v in t.items():
            sessions_annotated_shared[k] =  v

with open(f"{logs_online_non_shared_view_first}sessions_12_10_2025_annotations_new_ref_only_4.json", encoding="utf-8") as f:
    tmp = json.load(f)
    sessions_annotated_non_shared = {}
    for t in tmp:
        for k, v in t.items():
            sessions_annotated_non_shared[k] =  v

In [ ]:
import pandas as pd  
import matplotlib.pyplot as plt  
import numpy as np  

references = [
"Definite Description",
"Demonstrative Pronoun",
"Personal Pronoun",
# "Ambiguous Description",
"New Label",
"Ellipsis",
# "ambiguous"
]
def compare_senders_labels_barchart(sessions, human_sessions, category, role="human_helper", scope="Human"):  
    rows = [] 
    for session, data in sessions.items():
        if not session in human_sessions:
            continue
            
        if role == "human_helper":
            roles = {"helper": "Human", "worker": "AI"}
        else:
            roles = {"helper": "AI", "worker": "Human"}
            
        puzzles = data["puzzles"]
        for pid, data in puzzles.items():  
            if pid in ["6", "7", "8", "9"]:
                continue
            for msg, anns in zip(data["messages"], data["annotations"]):  
                # print(anns)
                if isinstance(anns, str):
                    continue
                for ann in anns["annotations"]:
                    label_val = ann[category]
                    if ann["referent_phrase"] == "None" and ann["referent"] != "Action":
                        continue
                    
                    # if category == "grounding_act" and not label_val in grounding:
                    #     continue
                    if category == "reference_type" and not label_val in references:
                        continue

                    rows.append({  
                        "puzzle": pid,  
                        "view": data["puzzle_details"]["config"]["puzzle_group"],
                        "label": label_val  
                    }) 
  
    df = pd.DataFrame(rows)  
  
    counts = df.groupby(["puzzle", "view", "label"]).size().unstack(fill_value=0) 
    counts2 = df.groupby(["puzzle", "view"]).size().unstack(fill_value=0) 
  
    puzzles_sorted = sorted(df["puzzle"].unique())  
    labels = list(counts.columns)  
  
    plt.figure(figsize=(12,5))  
  
    x = np.arange(len(puzzles_sorted))  
    width = 0.35 / len(labels)  
  
    for i, label in enumerate(labels):  
  
        vals_shared = []  
        vals_non_shared = []  
        
        for p in puzzles_sorted:  

            if (p, "Shared view") in counts.index: 
                s = counts2.loc[(p, f"Shared view")]
                vals_shared.append(counts.loc[(p, f"Shared view")].get(label, 0)/s)  
            else:  
                vals_shared.append(0)  
  
            if (p, "Non-shared view") in counts.index: 
                s = counts2.loc[(p, f"Non-shared view")]
                vals_non_shared.append(counts.loc[(p, f"Non-shared view")].get(label, 0)/s)  
            else:  
                vals_non_shared.append(0) 

          
        plt.bar(x + i*width - width*len(labels)/2 - 0.01,  
                vals_shared, width=width, label=f"{label} - Shared view")  
  
        plt.bar(x + i*width + width*len(labels)/2 + 0.01,  
                vals_non_shared, width=width, label=f"{label} - Non-shared view")  
        

    role_str = role.replace("_", " ")
    plt.xticks(x, puzzles_sorted)  
    plt.xlabel("Puzzle")  
    plt.ylabel("Count")  
    plt.title(f"{scope} {category} with {role_str}")  
    plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")  
    plt.tight_layout()  
    # plt.savefig(f"{category}_annotations_{worker}_worker_{helper}_helper_non_shared_view_first.png", dpi=300, bbox_inches="tight")  
    plt.show()   
    
 
# compare_senders_labels_barchart(sessions_annotated_shared|sessions_annotated_non_shared, human_worker_shared+human_worker_non_shared, "referent", role="human_worker", scope="Human")
# compare_senders_labels_barchart(sessions_annotated_shared|sessions_annotated_non_shared, human_worker_shared+human_worker_non_shared, "referent", role="human_worker", scope="AI")

# compare_senders_labels_barchart(sessions_annotated_shared|sessions_annotated_non_shared, human_helper_shared+human_helper_non_shared, "referent", role="human_helper", scope="Human")
# compare_senders_labels_barchart(sessions_annotated_shared|sessions_annotated_non_shared, human_helper_shared+human_helper_non_shared, "referent", role="human_helper", scope="AI")

# compare_senders_labels_barchart(sessions_annotated_non_shared, human_helper_non_shared, "reference_type", role="human_helper", scope="Human")
# compare_senders_labels_barchart(sessions_annotated_non_shared, human_worker_non_shared, "reference_type", role="human_worker", scope="AI")
# compare_senders_labels_barchart(sessions_annotated_non_shared, human_helper_non_shared, "reference_type", role="human_helper", scope="AI")
# compare_senders_labels_barchart(sessions_annotated_non_shared, human_worker_non_shared, "reference_type", role="human_worker", scope="AI")
# compare_senders_labels_barchart(sessions_annotated_non_shared, human_worker_non_shared, "reference_type", role="human_worker", scope="AI")

In [ ]:
import pandas as pd  
import matplotlib.pyplot as plt  
import numpy as np  
  
# grounding = ["clarify","correct","accept"]  
# references = ["definite","descriptive","pronoun"]  
# vocab = ["adopts_partner","introduces_new"]  
references = [
"Definite Description",
"Demonstrative Pronoun",
"Personal Pronoun",
"Ambiguous Description",
"New Label",
"Ellipsis"
# "ambiguous"
]
def compare_senders_labels_linechart(sessions, human_worker, human_helper, category, worker="Human", view="shared"):  
  
    rows = []  
    for session, data in sessions.items():  
        if worker == "Human":  
            helper = "AI"  
            if session not in human_worker:  
                continue  
        else:  
            helper = "Human"  
            if session not in human_helper:  
                continue  
  
        puzzles = data["puzzles"]  
        for pid, data in puzzles.items():  
            if pid in ["6", "7", "8", "9"]:
                continue
            for msg, anns in zip(data["messages"], data["annotations"]):  
                # print(anns)
                if isinstance(anns, str):
                    continue
                for ann in anns["annotations"]:
                    label_val = ann[category]
                    
                    if ann["referent_phrase"] == "None" and ann["referent"] != "Action":
                        continue
                    
                    # if category == "grounding_act" and not label_val in grounding:
                    #     continue
                    if category == "reference_type" and not label_val in references:
                        continue
                    # if category == "vocabulary_alignment" and not label_val in vocab:
                    #     continue
                    if label_val is None or label_val == "none":  
                        continue   # skip None labels  
                    if "Pronoun" in label_val:
                        label_val = "Pronoun"
                    if "Description" in label_val:
                        label_val = "Description"
                    rows.append({  
                        "puzzle": pid,  
                        "sender": msg["sender"],  
                        "label": label_val  
                    })  
  
    df = pd.DataFrame(rows)  
  
    counts = df.groupby(["puzzle", "sender", "label"]).size().unstack(fill_value=0)  
    totals = df.groupby(["puzzle", "sender"]).size()  
  
    puzzles_sorted = sorted(df["puzzle"].unique())  
    labels = list(counts.columns)  
  
    x = np.arange(len(puzzles_sorted))  
  
    # _, ax = plt.figure(figsize=(8, 5))
    fig, ax = plt.subplots(figsize=(8, 5))  
      
    for label in labels:  
        helper_vals = []  
      
        for p in puzzles_sorted:  
            if (p, "helper") in totals.index:  
                helper_vals.append(  
                    counts.loc[(p, "helper")].get(label, 0) /  
                    1#totals.loc[(p, "helper")]  
                )  
            else:  
                helper_vals.append(0)  
      
        plt.plot(  
            x,  
            helper_vals,  
            marker="o",  
            linewidth=2,  
            label=label  
        )  
      
    # plt.xticks(x, puzzles_sorted)  
    x = ["practice", "1", "2", "3", "4"]
    plt.xticks([0,1,2,3,4], x)  
    ax.axvspan(0, 1, color='yellow', alpha=0.15, label='practice')
    # ax.set_ylim(0, 1)  
    plt.xlabel("Puzzle ID")  
    plt.ylabel("Proportion")  
    plt.title(f"{helper} Helper")  
      
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")  
    plt.tight_layout()  
    plt.savefig(f"{category}_annotations_{helper}_helper_{view}_view_first.png", dpi=300, bbox_inches="tight")
    plt.show()  

    x = np.arange(len(puzzles_sorted))  
  
    # plt.figure(figsize=(8, 5)) 
    fig, ax = plt.subplots(figsize=(8, 5))
      
    for label in labels:  
        worker_vals = []  
      
        for p in puzzles_sorted:  
            if (p, "worker") in totals.index:  
                worker_vals.append(  
                    counts.loc[(p, "worker")].get(label, 0) /  
                    1#totals.loc[(p, "worker")]  
                )  
            else:  
                worker_vals.append(0)  
      
        plt.plot(  
            x,  
            worker_vals,  
            marker="o",  
            linewidth=2,  
            label=label  
        )  
      
    # plt.xticks(x, puzzles_sorted)  
    x = ["practice", "1", "2", "3", "4"]
    plt.xticks([0,1,2,3,4], x)  
    ax.axvspan(0, 1, color='yellow', alpha=0.15, label='practice')
    # ax.set_ylim(0, 1)  
    plt.xlabel("Puzzle ID")  
    plt.ylabel("Proportion")  
    plt.title(f"{worker} Worker")  
      
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")  
    plt.tight_layout()  
    plt.savefig(f"{category}_annotations_{worker}_worker_{view}_view_first.png", dpi=300, bbox_inches="tight")
    plt.show()  
  
  
# examples  
compare_senders_labels_linechart(sessions_annotated_shared, human_worker_shared, human_helper_shared, "reference_type", worker="Human", view="shared")  
compare_senders_labels_linechart(sessions_annotated_non_shared, human_worker_non_shared, human_helper_non_shared, "reference_type", worker="Human", view="non_shared") 

compare_senders_labels_linechart(sessions_annotated_shared, human_worker_shared, human_helper_shared, "reference_type", worker="AI", view="shared")  
compare_senders_labels_linechart(sessions_annotated_non_shared, human_worker_non_shared, human_helper_non_shared, "reference_type", worker="AI", view="non_shared") 

In [ ]:
import nltk  
from nltk import word_tokenize, pos_tag 
from nltk import word_tokenize, pos_tag, RegexpParser  
grammar = r"""  
  NP: {<DT>?<JJ.*>*<NN.*>+}  
"""  
  
text = "Move the small red block on the left side."  
tokens = word_tokenize(text)  
tagged = pos_tag(tokens)  
  
chunker = RegexpParser(grammar)  
tree = chunker.parse(tagged)  
  
noun_phrases = [  
    " ".join(word for word, pos in subtree.leaves())  
    for subtree in tree  
    if isinstance(subtree, nltk.Tree) and subtree.label() == "NP"  
]  
  
print(noun_phrases)  

In [ ]:
import matplotlib.pyplot as plt 

def plot_vocab_size(sessions, human_worker, human_helper, view="shared"):
    rows = []
    for session, data in sessions.items():  
        puzzles = data["puzzles"]  
        messages = []
        referents = set()
        
        for pid, data in puzzles.items():  
            if pid in ["6", "7", "8", "9"]:
                continue
            messages = []
            pids = []
            referents_from_AI = set()     
            referents_from_human = set()
            for i, (msg, anns) in enumerate(zip(data["messages"], data["annotations"])):  
                messages.append(msg)
                pids.append(pid)
                if isinstance(anns, str):
                    continue
                
                for ann in anns["annotations"]:
                    # if "Description" not in ann["reference_type"]:
                    #     continue
                    if ann["referent_phrase"] == "None" or ann["referent"] == "Action":
                        continue
                    else:
                        referent = ann["referring_phrase"].lower()                 
                        if session in human_worker:
                            if msg["sender"] == "worker":  
                                referents_from_human.add(referent)
                            else:
                                referents.add(referent)
                                referents_from_AI.add(referent)
                        elif session in human_helper:
                            if msg["sender"] == "helper":
                                referents_from_human.add(referent)                        
                            else:
                                referents.add(referent)
                                referents_from_AI.add(referent)
      
            rows.append({  
                "puzzle": pid, 
                "union_vocab_size": len(referents_from_human | referents_from_AI),
                "joint_vocab_size": len(referents_from_human & referents_from_AI),
                "AI_vocab_size": len(referents_from_AI),
                "human_vocab_size": len(referents_from_human)
            })  
    
        # break

    df = pd.DataFrame(rows)  
    grouped = (  
        df.groupby('puzzle')[['union_vocab_size','joint_vocab_size', 'AI_vocab_size', 'human_vocab_size']]  
          .mean()  
    )   
    plt.figure(figsize=(8, 5))  
    # plt.plot(grouped.index, grouped['union_vocab_size'], marker='o', label='Union')    
    plt.plot(grouped.index, grouped['joint_vocab_size'], marker='o', label='Joint')  
    plt.plot(grouped.index, grouped['AI_vocab_size'], marker='o', label='AI')  
    plt.plot(grouped.index, grouped['human_vocab_size'], marker='o', label='Human')  
      
    plt.xlabel('Puzzle')  
    plt.ylabel('Vocabulary Size')  
    plt.title(f'Vocabulary size with {view.replace("_", " ")} view')  
    plt.legend()  
    plt.grid(True)  
    plt.savefig(f"vocab_sizes_{view}_view_first.png", dpi=300, bbox_inches="tight")
    
    plt.show()  

import pandas as pd  
import numpy as np  
import matplotlib.pyplot as plt 
import re

def extract_ngrams_atomic(text, n):  
    tokens = re.findall(r"$[^)]*$|$[^$]*\]|\{[^}]*\}|\w+", text.lower())  
      
    return [  
        " ".join(tokens[i:i+n])  
        for i in range(len(tokens) - n + 1)  
    ]  

def get_NPs(text):
    grammar = r"""  
      NP: {<DT>?<JJ.*>*<NN.*>+}  
    """    
    tokens = word_tokenize(text.lower())  
    tagged = pos_tag(tokens)  
      
    chunker = RegexpParser(grammar)  
    tree = chunker.parse(tagged)  
      
    noun_phrases = [  
        " ".join(word for word, pos in subtree.leaves())  
        for subtree in tree  
        if isinstance(subtree, nltk.Tree) and subtree.label() == "NP"  
    ] 
    return noun_phrases#" ".join(noun_phrases).split()
      
def plot_vocab_size_stacked(  
    sessions,  
    human_worker,  
    human_helper,  
    view="shared"  
):  
    rows = []  

    # referents_from_AI = set()  
    # referents_from_human = set()
    for session, session_data in sessions.items():  
        puzzles = session_data["puzzles"]  

         
        for pid, puzzle_data in puzzles.items():  
  
            # skip unwanted puzzles  
            if pid in ["6", "7", "8", "9"]:  
                continue  
  
            referents_from_AI = set()  
            referents_from_human = set() 
  
            for msg, anns in zip(  
                puzzle_data["messages"],  
                puzzle_data["annotations"]  
            ):  
                if isinstance(anns, str):  
                    continue  
  
                # for ann in anns["annotations"]:  
                #     # if ann["referent_phrase"] == "None":  
                #     #     continue  
                #     # if ann["referent"] == "Action":  
                #     #     continue
                #     # if ann["referent"] == "Location":  
                #     #     continue
  
                #     referent = ann["referring_phrase"].lower() 
                #     # referent = re.sub(r"[^\w\s$$$$\{\}]", "", referent.lower()) 
                #     # referent = ann["referent_phrase"].lower() 
  
                for lex in get_NPs(msg["message"]):#referent.split():  
                    if session in human_worker:  
                        if msg["sender"] == "worker":  
                            referents_from_human.add(lex)  
                        else:  
                            referents_from_AI.add(lex)  
  
                    elif session in human_helper:  
                        if msg["sender"] == "helper":  
                            referents_from_human.add(lex)  
                        else:  
                            referents_from_AI.add(lex)  

            # print( referents_from_human & referents_from_AI,  referents_from_human - referents_from_AI, referents_from_AI - referents_from_human)
            rows.append({  
                "puzzle": pid,  
                "joint_vocab_size": len(  
                    referents_from_human & referents_from_AI  
                ),  
                "human_only_vocab_size": len(  
                    referents_from_human - referents_from_AI  
                ),  
                "AI_only_vocab_size": len(  
                    referents_from_AI - referents_from_human  
                )  
            })  
  
    # build dataframe  
    df = pd.DataFrame(rows)  
  
    # aggregate across sessions  
    grouped = (  
        df.groupby("puzzle")[  
            ["joint_vocab_size",  
             "human_only_vocab_size",  
             "AI_only_vocab_size"]  
        ]  
        .mean()  
        .sort_index()  
    )  

    # stacked bar plot  
    x = ["practice", "1", "2", "3", "4"]#np.arange(len(grouped.index))  
    # normalize  
    prop = grouped.div(grouped.sum(axis=1), axis=0)  
      
    plt.figure(figsize=(9, 5))  

    plt.bar(x, prop["human_only_vocab_size"], label="Human-only", color="green") 
    
    plt.bar(  
        x,  
        prop["joint_vocab_size"],  
        bottom=prop["human_only_vocab_size"],  
        label="Shared",  
        color="blue"  
    )  
    plt.bar(  
        x,  
        prop["AI_only_vocab_size"],  
        bottom=prop["human_only_vocab_size"] + prop["joint_vocab_size"],  
        label="AI-only",  
        color="orange"  
    )  
      
    plt.ylabel("Proportion of Vocabulary")  
    plt.title(f"Proportion of Shared vs Private Vocabulary with {view.replace('_', ' ')} view")  
  
    plt.legend(bbox_to_anchor=(1.05,1), loc="upper left") 
    plt.grid(axis="y", alpha=0.3)  
  
    plt.tight_layout()  
    plt.savefig(  
        f"vocab_sizes_{view}_stacked_union.png",  
        dpi=300,  
        bbox_inches="tight"  
    )  
    plt.show()  
    

# plot_vocab_size_stacked(  
#     sessions_annotated_shared,  
#     human_worker_shared,  
#     human_helper_shared,  
#     view="shared"  
# )

# plot_vocab_size_stacked(  
#     sessions_annotated_non_shared,  
#     human_worker_non_shared,  
#     human_helper_non_shared,  
#     view="non_shared"  
# )  

In [ ]:
with open(f"{logs_online_shared_view_first}sessions_12_08_2025_annotations_new_grounding_acts2.json", encoding="utf-8") as f:
    tmp = json.load(f)
    sessions_grounding_shared = {}
    for t in tmp:
        for k, v in t.items():
            sessions_grounding_shared[k] =  v

with open(f"{logs_online_non_shared_view_first}sessions_12_10_2025_annotations_new_grounding2.json", encoding="utf-8") as f:
    tmp = json.load(f)
    sessions_grounding_non_shared = {}
    for t in tmp:
        for k, v in t.items():
            sessions_grounding_non_shared[k] =  v

In [ ]:
import pandas as pd  
import matplotlib.pyplot as plt  
import numpy as np  
  
groundings = [
    "Acceptance",
    "Presentation",
    "Clarification",
    "Correction",
    "Repair",
    "Followup"
]

def compare_senders_labels_grounding_chart(sessions, human_worker, human_helper, worker="Human", view="shared"):  
    rows = []  
    for session, data in sessions.items():  
        if worker == "Human":  
            helper = "AI"  
            if session not in human_worker:  
                continue  
        else:  
            helper = "Human"  
            if session not in human_helper:  
                continue  
  
        puzzles = data["puzzles"]  
        for pid, data in puzzles.items():  
            if pid in ["6", "7", "8", "9"]:
                continue
            for msg, anns in zip(data["messages"], data["annotations"]):  
                # print(anns)
                if isinstance(anns, str):
                    continue
                seen = set()
                for ann in anns["annotations"]:
                    label_val = ann["grounding_type"]
                    # if ann["referent_phrase"] == "None" and label_val != "Presentation":
                        # continue
                    
                    if not label_val in groundings:
                        continue

                    # if label_val == "Acceptance" and msg["sender"] == "helper":
                        # continue

                    if label_val == "Followup":
                        label_val = "Acceptance"
                    if label_val == "Correction":
                        label_val = "Repair"
                        
                    if label_val in seen:
                        continue

                    seen.add(label_val)
                    rows.append({  
                        "puzzle": pid,  
                        "sender": msg["sender"],  
                        "label": label_val  
                    })  
                    # 
  
    df = pd.DataFrame(rows)
    # print(df)
  
    counts = df.groupby(["puzzle", "sender", "label"]).size().unstack(fill_value=0)  
    totals = df.groupby(["puzzle", "sender"]).size()  
  
    puzzles_sorted = sorted(df["puzzle"].unique())  
    labels = list(counts.columns)  

    x = np.arange(len(puzzles_sorted))  
  
    # _, ax = plt.figure(figsize=(8, 5))
    fig, ax = plt.subplots(figsize=(8, 5))  
      
    for label in labels:  
        helper_vals = []  
      
        for p in puzzles_sorted:  
            if (p, "helper") in totals.index:  
                helper_vals.append(  
                    counts.loc[(p, "helper")].get(label, 0) /  
                    1#totals.loc[(p, "helper")]  
                )  
            else:  
                helper_vals.append(0)  
      
        plt.plot(  
            x,  
            helper_vals,  
            marker="o",  
            linewidth=2,  
            label=label  
        )  
      
    # plt.xticks(x, puzzles_sorted)  
    x = ["practice", "1", "2", "3", "4"]
    plt.xticks([0,1,2,3,4], x)  
    ax.axvspan(0, 1, color='yellow', alpha=0.15, label='practice')
    handles, plot_labels = ax.get_legend_handles_labels()  
    order = [2,1,3,0,4]
    ax.legend([handles[i] for i in order], [plot_labels[i] for i in order], bbox_to_anchor=(1.05, 1), loc="upper left") 

    # ax.set_ylim(0, 1)  
    plt.xlabel("Puzzle ID")  
    plt.ylabel("Count")  
    plt.title(f"{helper} Helper")  
      
    plt.tight_layout()  
    plt.savefig(f"grounding_annotations_{helper}_helper_{view}_view_first.png", dpi=300, bbox_inches="tight")
    plt.show()  

    x = np.arange(len(puzzles_sorted))  
    plt.clf()
    ax.clear()
  
    # plt.figure(figsize=(8, 5)) 
    fig, ax = plt.subplots(figsize=(8, 5))
      
    for label in labels:  
        worker_vals = []  
      
        for p in puzzles_sorted:  
            if (p, "worker") in totals.index:  
                worker_vals.append(  
                    counts.loc[(p, "worker")].get(label, 0) /  
                    1#totals.loc[(p, "worker")]  
                )  
            else:  
                worker_vals.append(0)  
      
        plt.plot(  
            x,  
            worker_vals,  
            marker="o",  
            linewidth=2,  
            label=label  
        )  
      
    # plt.xticks(x, puzzles_sorted)  
    x = ["practice", "1", "2", "3", "4"]
    plt.xticks([0,1,2,3,4], x)  
    ax.axvspan(0, 1, color='yellow', alpha=0.15, label='practice')
    handles, plot_labels = ax.get_legend_handles_labels()  
    order = [2,1,3,0,4]
    ax.legend([handles[i] for i in order], [plot_labels[i] for i in order], bbox_to_anchor=(1.05, 1), loc="upper left") 

    # ax.set_ylim(0, 1)  
    plt.xlabel("Puzzle ID")  
    plt.ylabel("Count")  
    plt.title(f"{worker} Worker")  
    
    plt.tight_layout()  
    plt.savefig(f"grounding_annotations_{worker}_worker_{view}_view_first.png", dpi=300, bbox_inches="tight")
    plt.show()

  
  
# examples  
# compare_senders_labels_grounding_chart(sessions_grounding_shared, human_worker_shared, human_helper_shared, worker="Human", view="shared") 
# compare_senders_labels_grounding_chart(sessions_grounding_shared, human_worker_shared, human_helper_shared, worker="AI", view="shared") 

# compare_senders_labels_grounding_chart(sessions_grounding_non_shared, human_worker_non_shared, human_helper_non_shared, worker="Human", view="non_shared") 
# compare_senders_labels_grounding_chart(sessions_grounding_non_shared, human_worker_non_shared, human_helper_non_shared, worker="AI", view="non_shared") 